# cheaseBS iterative run
Edit the paths and options in the **Configuration** cell, then run all cells.

## Configuration

In [1]:
import sys, os

# --- repo & binary paths ---------------------------------------------------
REPO_DIR     = "/global/u1/j/joeschm/simulators/cheaseBS"
CHEASE_BIN   = "/global/u1/j/joeschm/simulators/chease/src-f90/chease"  # adjust if needed

# --- input files -----------------------------------------------------------
DATA_DIR = "/global/homes/j/joeschm/data/ST_research/NSTXU_discharges/129015"
EQDSK = os.path.join(DATA_DIR, "g129015.00409_450")     # your geqdsk

# active profiles (the ones you want to run with)
PROF_E       = os.path.join(DATA_DIR, "profiles_e")
PROF_I       = os.path.join(DATA_DIR, "profiles_i")
PROF_Z       = os.path.join(DATA_DIR, "profiles_z")

# reference profiles (used only to build the baseline decomposition)
# set to None to use the same files as the active profiles
REF_PROF_E   = PROF_E
REF_PROF_I   = PROF_I
REF_PROF_Z   = PROF_Z

# --- output ----------------------------------------------------------------
BASELINE_DIR = "/global/homes/j/joeschm/projects/cheaseBS_tests/runs/baseline"
OUTPUT_DIR   = "/global/homes/j/joeschm/projects/cheaseBS_tests/runs/iterative"

# --- solver options --------------------------------------------------------
REPLAY_REPR  = "istar"   # "jparallel" (DIII-D) or "istar" (NSTX)
COORDINATE   = "rhot"        # "rhot" or "rhop"
MAX_ITER     = 8
TOL_IP_REL   = 1e-3
TOL_BS       = 1e-3
TOL_Q        = 1e-3
TOL_A        = 1e-4
ENFORCE_QSPEC    = False      # set False for NSTX / non-DIII-D
REBUILD_BASELINE = True      # set False to reuse an existing baseline
OVERWRITE        = True

## Build argument list and run

In [2]:
import subprocess

script = os.path.join(REPO_DIR, "run_chease_iterative_profiles.py")

args = [
    sys.executable, script,
    "--eqdsk",              EQDSK,
    "--electron-profile",   PROF_E,
    "--deuterium-profile",  PROF_I,
    "--carbon-profile",     PROF_Z,
    "--baseline-dir",       BASELINE_DIR,
    "--output-dir",         OUTPUT_DIR,
    "--chease-binary",      CHEASE_BIN,
    "--replay-representation", REPLAY_REPR,
    "--coordinate",         COORDINATE,
    "--max-iter",           str(MAX_ITER),
    "--tol-ip-rel",         str(TOL_IP_REL),
    "--tol-bs",             str(TOL_BS),
    "--tol-q",              str(TOL_Q),
    "--tol-a",              str(TOL_A),
]

# optional reference profiles
if REF_PROF_E: args += ["--reference-electron-profile", REF_PROF_E]
if REF_PROF_I: args += ["--reference-deuterium-profile", REF_PROF_I]
if REF_PROF_Z: args += ["--reference-carbon-profile",   REF_PROF_Z]

if ENFORCE_QSPEC:    args.append("--enforce-qspec")
else:                args.append("--no-enforce-qspec")
if REBUILD_BASELINE: args.append("--rebuild-baseline")
if OVERWRITE:        args.append("--overwrite")

print("Running:\n", " \\ ".join(args), "\n")

result = subprocess.run(args, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"cheaseBS exited with code {result.returncode}")

Running:
 /global/homes/j/joeschm/.conda/envs/TPED/bin/python \ /global/u1/j/joeschm/simulators/cheaseBS/run_chease_iterative_profiles.py \ --eqdsk \ /global/homes/j/joeschm/data/ST_research/NSTXU_discharges/129015/g129015.00409_450 \ --electron-profile \ /global/homes/j/joeschm/data/ST_research/NSTXU_discharges/129015/profiles_e \ --deuterium-profile \ /global/homes/j/joeschm/data/ST_research/NSTXU_discharges/129015/profiles_i \ --carbon-profile \ /global/homes/j/joeschm/data/ST_research/NSTXU_discharges/129015/profiles_z \ --baseline-dir \ /global/homes/j/joeschm/projects/cheaseBS_tests/runs/baseline \ --output-dir \ /global/homes/j/joeschm/projects/cheaseBS_tests/runs/iterative \ --chease-binary \ /global/u1/j/joeschm/simulators/chease/src-f90/chease \ --replay-representation \ istar \ --coordinate \ rhot \ --max-iter \ 8 \ --tol-ip-rel \ 0.001 \ --tol-bs \ 0.001 \ --tol-q \ 0.001 \ --tol-a \ 0.0001 \ --reference-electron-profile \ /global/homes/j/joeschm/data/ST_research/NSTXU_disc

KeyboardInterrupt: 

## Quick convergence check

In [ ]:
import glob, json

# look for a convergence summary JSON if the script writes one
summaries = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True))
for s in summaries:
    print(s)
    with open(s) as f:
        print(json.dumps(json.load(f), indent=2))
    print()